# Sample CatBoost Training Notebook

This notebook shows a simple local training flow and saves artifacts to:

- `models/training/catboost/<symbol>/<timeframe>/<version>/`

Optionally it can also copy the trained model to:

- `models/live/catboost/<symbol>_<timeframe>.cbm`

In [ ]:
from __future__ import annotations

import json
import shutil
from datetime import datetime, UTC
from pathlib import Path

import pandas as pd
from catboost import CatBoostClassifier

In [ ]:
# --- User inputs ---
DATA_PATH = Path("data/meta_train.csv")
TARGET_COL = "target"
SYMBOL = "META"
TIMEFRAME = "1m"
FEATURE_COLS: list[str] | None = None  # set list explicitly, or keep None to auto-pick
VAL_SIZE = 0.2
ITERATIONS = 300
LEARNING_RATE = 0.05
DEPTH = 6
PROMOTE_TO_LIVE = False

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "training" else Path.cwd().resolve()
DATA_PATH = (REPO_ROOT / DATA_PATH).resolve() if not DATA_PATH.is_absolute() else DATA_PATH
print("Repo root:", REPO_ROOT)
print("Data path:", DATA_PATH)

In [ ]:
df = pd.read_csv(DATA_PATH)
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column not found: {TARGET_COL}")

if FEATURE_COLS is None:
    feature_cols = [c for c in df.columns if c != TARGET_COL]
else:
    missing = [c for c in FEATURE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    feature_cols = FEATURE_COLS

df = df.dropna(subset=feature_cols + [TARGET_COL]).reset_index(drop=True)
if len(df) < 50:
    raise ValueError("Not enough rows after dropna; need at least 50 rows.")

split_idx = int(len(df) * (1 - VAL_SIZE))
if split_idx <= 0 or split_idx >= len(df):
    raise ValueError("VAL_SIZE left no data for train or validation.")

train_df = df.iloc[:split_idx].copy()
val_df = df.iloc[split_idx:].copy()

x_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]
x_val = val_df[feature_cols]
y_val = val_df[TARGET_COL]

print("Rows -> train:", len(train_df), "val:", len(val_df))
print("Features:", len(feature_cols))

In [ ]:
model = CatBoostClassifier(
    iterations=ITERATIONS,
    learning_rate=LEARNING_RATE,
    depth=DEPTH,
    eval_metric="AUC",
    loss_function="Logloss",
    verbose=50,
)

model.fit(x_train, y_train, eval_set=(x_val, y_val), use_best_model=True)

In [ ]:
version = datetime.now(UTC).strftime("v%Y-%m-%d_%H%M%S")
artifact_dir = (
    REPO_ROOT
    / "models"
    / "training"
    / "catboost"
    / SYMBOL
    / TIMEFRAME
    / version
)
artifact_dir.mkdir(parents=True, exist_ok=False)

model_path = artifact_dir / "model.cbm"
model.save_model(model_path)

val_probs = model.predict_proba(x_val)[:, 1]
metrics = {
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "val_probability_mean": float(val_probs.mean()),
    "best_iteration": int(model.get_best_iteration()),
    "best_score": model.get_best_score(),
}
metadata = {
    "symbol": SYMBOL,
    "timeframe": TIMEFRAME,
    "target_col": TARGET_COL,
    "feature_cols": feature_cols,
    "data_path": str(DATA_PATH),
    "version": version,
}
train_config = {
    "iterations": ITERATIONS,
    "learning_rate": LEARNING_RATE,
    "depth": DEPTH,
    "val_size": VAL_SIZE,
}

(artifact_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
(artifact_dir / "feature_schema.json").write_text(json.dumps(metadata, indent=2))
(artifact_dir / "train_config.json").write_text(json.dumps(train_config, indent=2))

if PROMOTE_TO_LIVE:
    live_dir = REPO_ROOT / "models" / "live" / "catboost"
    live_dir.mkdir(parents=True, exist_ok=True)
    live_path = live_dir / f"{SYMBOL}_{TIMEFRAME}.cbm"
    shutil.copy2(model_path, live_path)
    print("Promoted to live:", live_path)

print("Saved training artifacts to:", artifact_dir)
print("Saved model:", model_path)
metrics